# EcoRewind 2.0 — Phase 1: cube, masks, blocks, folds (Colab, **CPU runtime**)

One-time setup: a GitHub token in Colab **Secrets** (key icon in the left bar) named `GITHUB_TOKEN`, with notebook access switched on. Everything reads the Phase 0 exports from Drive and writes results to `MyDrive/EcoRewind2/phase1/`. Expected run time: about 30–45 min, about 0.1 compute units.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/ecorewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt

### 1. Tests (must all pass before anything else)

In [ ]:
!python -m pytest -q tests

### 2. Check the pre-registered placebo windows against NOAA HURDAT2
Any ❌ or ⚠️ must be decided by you before Phase 2; nothing is changed automatically.

In [ ]:
D = '/content/drive/MyDrive/EcoRewind2'
!python scripts/check_storm_record.py --out {D}/phase1/storm_record.md

### 3. Copy exports to the VM disk, then build the cubes
Cubes are built on the VM (`/content/cubes`, ~10 GB). They are **not** saved to Drive by default (Drive space); rebuilding takes ~15 min. Set `SAVE_CUBES = True` only if you have the space.

In [ ]:
!python scripts/copy_exports.py --src {D}/../EcoRewind2_exports --dst /content/exports
SAVE_CUBES = False
ZIP = f'--zip-to {D}/cubes' if SAVE_CUBES else ''
!python scripts/build_cube.py --exports /content/exports --out /content/cubes {ZIP} | grep -v ': optical'
!du -sh /content/cubes/*

### 4. Static layers, blocks, folds, buffers, normalisation, fold maps

In [ ]:
!python scripts/make_folds.py --cubes /content/cubes --out {D}/phase1
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{D}/phase1/phase1_summary.md').read()))
for s in ('everglades', 'mississippi'):
    display(Image(f'{D}/phase1/figures/folds_{s}.png'))

Send `phase1/phase1_summary.md`, `phase1/storm_record.md` and the two fold maps. Then **Runtime → Disconnect and delete runtime** (it bills while idle).

In [ ]:
!git -C /content/ecorewind2 pull -q
%cd /content/ecorewind2
!python scripts/check_storm_record.py --out {D}/phase1/storm_record.md
!python scripts/make_folds.py --cubes /content/cubes --out {D}/phase1